Getting and Fixing Dataset

In [1]:
import kagglehub
import pandas as pd
import numpy as np
import os

# Download latest version
path = kagglehub.dataset_download("redwankarimsony/heart-disease-data")

print("Path to dataset files:", path)

files = os.listdir(path)
print("Files:", files)

file_path = os.path.join(path, files[0])  #picking the CSV file
df = pd.read_csv(file_path)

print(df.head())
print(df.info())

Using Colab cache for faster access to the 'heart-disease-data' dataset.
Path to dataset files: /kaggle/input/heart-disease-data
Files: ['heart_disease_uci.csv']
   id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   
3          normal   187.0  False      3.5  downsloping  0.0   
4  lv hypertrophy   172.0  False      1.4    upsloping  0.0   

              

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os

#Check missing values in some of the columns that will be dropped
print("\nMissing values before dropna():\n", df.isnull().sum())
df = df.dropna()

#Because we are only targetting whether an individual has a heart disease or not, I am
#changing the numbers in the data set because they use numbers to describe the severity of the disease (1-4) and 0 if they don't have a disease
#I am turning 1-4 into only 1 to describe that they have a disease and 0 as no disease present
df['target'] = df['num'].apply(lambda x: 1 if x > 0 else 0)

#Dropping the 'num' column given by the dataset as it's no longer needed
df = df.drop(columns=['num'])

df = df.drop(columns=['id', 'dataset'])

# Distribution of th new 'target' column
print("\nTarget column value counts:\n", df['target'].value_counts())

# Separate features (X) and target (y)
X = df.drop(columns=['target'])
y = df['target']

#Checking the missing values in the columns after being dropped
print("Missing values after df.dropna():\n", df.isnull().sum())


Missing values before dropna():
 id            0
age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64

Target column value counts:
 target
0    160
1    139
Name: count, dtype: int64
Missing values after df.dropna():
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


Training, Splitting and Evaluating Data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = df.drop(columns=['target'])
y = df['target']

# Identify categorical columns for one-hot encoding
categorical_cols = X.select_dtypes(include=['object', 'bool']).columns

# Apply one-hot encoding to categorical features
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)



#Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler= StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=5000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

#Evaluating and printin the accuracy scores, classification report and confusion matriz
accuracy = accuracy_score(y_test, y_pred)
classification_report_str = classification_report(y_test, y_pred)
confusion_matrix_str = str(confusion_matrix(y_test, y_pred))
print(f"Accuracy: {accuracy}")
print("Classification Report:\n", classification_report_str)
print("Confusion Matrix:\n", confusion_matrix_str)


Accuracy: 0.9
Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.91      0.91        35
           1       0.88      0.88      0.88        25

    accuracy                           0.90        60
   macro avg       0.90      0.90      0.90        60
weighted avg       0.90      0.90      0.90        60

Confusion Matrix:
 [[32  3]
 [ 3 22]]


Estimating likelihood of disease

In [ ]:
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("\nSample probabilities:\n", y_prob)


# showing the factors that can lead to heart disease
feature_names = X.columns
coefficients = model.coef_[0]

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
}).sort_values(by="Coefficient", key=abs, ascending=False)

print("\nLogistic Regresion Coefficients:\n", importance_df)



Sample probabilities:
 [0.98938111 0.03495919 0.57322637 0.86185156 0.11250107 0.91571252
 0.01096068 0.81035009 0.09283241 0.99114658 0.33406492 0.03260044
 0.0557957  0.99410988 0.31073233 0.73098731 0.14923638 0.73560584
 0.0058204  0.52456494 0.79022467 0.22810772 0.76640283 0.04032105
 0.41082159 0.19072441 0.71091952 0.0588952  0.01224196 0.10767804
 0.30224699 0.05215841 0.70059987 0.99473555 0.66238644 0.1612549
 0.46993054 0.4162715  0.06375657 0.01120851 0.04747275 0.05920035
 0.97469249 0.98538298 0.32997342 0.04548373 0.95197418 0.99135137
 0.00275282 0.72847694 0.02691363 0.78836889 0.11024942 0.9706931
 0.03022145 0.75176487 0.93114283 0.03400406 0.20215117 0.45102506]

Logistic Regresion Coefficients:
                      Feature  Coefficient
5                         ca     1.094731
8             cp_non-anginal    -0.792993
6                   sex_Male     0.688870
9          cp_typical angina    -0.519460
13                exang_True     0.449508
4                   

Prediction

In [ ]:
def predict_heart_disease(input_data):
    # Convert input_data to a DataFrame with correct feature names
    input_df = pd.DataFrame([input_data], columns=X.columns)
    input_scaled_array = scaler.transform(input_df)
    # Convert scaled array back to DataFrame with feature names
    input_scaled_df = pd.DataFrame(input_scaled_array, columns=X.columns)

    prediction = model.predict(input_scaled_df)
    probability = model.predict_proba(input_scaled_df)

    return prediction[0], probability[0][1]

sample_patient = X.iloc[0].values

pred, prob = predict_heart_disease(sample_patient)

print("\nPrediction (1 = disease, 0 = no disease):", pred)
print("Probability of heart disease:", prob)


Prediction (1 = disease, 0 = no disease): 0
Probability of heart disease: 0.1055903129864296


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
